In [1]:
import pandas as pd

In [2]:
df_pairs_bodega = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\support_analysis_bdg.xlsx")

In [3]:
df_pairs_cluster = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\support_analysis_clst.xlsx")

In [4]:
clusters = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\clusters_bdg.xlsx")

In [5]:
clusters_fc = clusters[clusters["UNE"] == "FARMACORP"]

In [6]:
df_pairs_bodega

,COD_BODEGA,CAT 4_A,CAT 4_B,NRO_FACTURAS_CAT4_A,NRO_FACTURAS_CAT4_B,NRO_FACTURAS_PAIR,NRO_FACTURAS_TOTAL_BDG,support_cat4_A,support_cat4_B,support_AB,confidence_AB_A,lift_ABA_B,Ventas
0,B101,ANTIRREUMATICOS NO ESTEROIDEOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,3435,2908,373,23689,0.145004,0.122757,0.015746,0.108588,0.884574,47325.03
1,B101,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIGRIPALES EXC.ANTIINFARINGITIS,3435,2620,361,23689,0.145004,0.110600,0.015239,0.105095,0.950224,51735.93
2,B101,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIBACTERIANOS PENICILINAS AMPLIO ESPECTRO,3435,593,197,23689,0.145004,0.025033,0.008316,0.057351,2.291035,21954.81
3,B101,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIHISTAMINICOS-ANTIALERGICOS,3435,655,86,23689,0.145004,0.027650,0.003630,0.025036,0.905477,15373.60
4,B101,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIACIDOS SOLOS,3435,1052,83,23689,0.145004,0.044409,0.003504,0.024163,0.544105,11203.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
183572,B901,OTRAS VITAMINAS SOLAS Y COMBINADAS,ANALGESICOS NARCOTICOS,159,281,1,23621,0.006731,0.011896,0.000042,0.006290,0.528742,11992.01
183573,B901,OTRAS VITAMINAS SOLAS Y COMBINADAS,ANGIOTENSINA-II ANTAGONISTAS SOLO,159,275,1,23621,0.006731,0.011642,0.000042,0.006290,0.540278,12935.45
183574,B901,OTRAS VITAMINAS SOLAS Y COMBINADAS,ANTIBACTERIANOS PENICILINAS MEDIO Y REDUCIDO E...,159,194,1,23621,0.006731,0.008213,0.000042,0.006290,0.765859,9776.75
183575,B901,OTRAS VITAMINAS SOLAS Y COMBINADAS,ANTIESPASMODICOS,159,199,1,23621,0.006731,0.008425,0.000042,0.006290,0.746616,5131.57


In [6]:
df_pairs_bodega = pd.merge(
    df_pairs_bodega, 
    clusters[['BODEGA', 'CLUSTER']], 
    left_on='COD_BODEGA',  
    right_on = 'BODEGA', 
    how='left'
    )

In [7]:
df_pairs_bodega['pair_key'] = df_pairs_bodega.apply(
    lambda x: '_'.join(sorted([x['CAT 4_A'], x['CAT 4_B'], x['CLUSTER']])),
    axis=1
)

df_pairs_cluster['pair_key'] = df_pairs_cluster.apply(
    lambda x: '_'.join(sorted([x['CAT 4_A'], x['CAT 4_B'], x['CLUSTER']])),
    axis=1
)

In [8]:
pairs_op = pd.merge(
    df_pairs_bodega,
    df_pairs_cluster,
    on='pair_key',
    suffixes=('_bdg', '_clst'),
    how='left'
)

In [9]:
pairs_op['new_support_B'] = pairs_op['confidence_AB_A_bdg']/pairs_op['lift_ABA_B_clst']

In [10]:
pairs_op['support_gap_B'] = pairs_op['new_support_B'] - pairs_op['support_cat4_B_bdg']

In [11]:
pairs_op['new_tickets'] = pairs_op['support_gap_B'] * pairs_op['NRO_FACTURAS_CAT4_A_bdg']

In [12]:
pairs_op['tkt_avg_B'] = pairs_op['Ventas']/pairs_op['NRO_FACTURAS_CAT4_B_bdg']

In [13]:
pairs_op['opportunity_B'] = pairs_op['tkt_avg_B']*pairs_op['new_tickets']*4

In [14]:
pairs_op['validation'] = ((pairs_op['opportunity_B'] > 0) & (pairs_op['lift_ABA_B_clst'] > 1.5)).astype(int)


In [21]:
pair_analysis = pairs_op[[
    'COD_BODEGA',
    'CLUSTER_bdg',

    'CAT 4_A_bdg',
    'CAT 4_B_bdg',
    'NRO_FACTURAS_CAT4_A_bdg',
    'NRO_FACTURAS_CAT4_B_bdg',
    'NRO_FACTURAS_PAIR_bdg',
    'NRO_FACTURAS_TOTAL_BDG',
    'support_cat4_A_bdg',
    'support_cat4_B_bdg',
    'support_AB_bdg',
    'confidence_AB_A_bdg',
    'lift_ABA_B_bdg',
    
    'CAT 4_A_clst',
    'CAT 4_B_clst',
    'NRO_FACTURAS_CAT4_A_clst',
    'NRO_FACTURAS_CAT4_B_clst',
    'NRO_FACTURAS_PAIR_clst',
    'NRO_FACTURAS_TOTAL_CLST',
    'support_cat4_A_clst',
    'support_cat4_B_clst',
    'support_AB_clst',
    'confidence_AB_A_clst',
    'lift_ABA_B_clst',
    
    'new_support_B',
    'support_gap_B',
    'new_tickets',
    'tkt_avg_B',
    'opportunity_B',
    'validation'
]].copy()

In [23]:
pair_analysis['pairs'] = pair_analysis['CAT 4_A_bdg'] + '|' + pair_analysis['CAT 4_B_bdg']

In [24]:
pairs_val = pair_analysis[pair_analysis['validation']==1]

In [46]:
best_opportunities = (
    pairs_val
    .groupby('pairs', as_index=False)
    .agg(opportunity_B=('opportunity_B', 'sum'))
).round(2)

In [47]:
best_opportunities

,pairs,opportunity_B
0,ANALGESICOS NARCOTICOS|AGENTES ANTIREUMATICOS ...,173.77
1,ANGIOTENSINA-II ANTAGONISTAS COMBINACIONES|ANA...,362.40
2,ANGIOTENSINA-II ANTAGONISTAS SOLO|ANGIOTENSINA...,62869.80
3,ANTIACIDOS SOLOS|ANTIACIDO CON ANTIFLATULENTO,10559.40
4,ANTIAGREGANTE PLAQUETARIO|ANALGESICOS NARCOTICOS,1079.85
...,...,...
619,VITAMINAS A+D ASOCIACIONES SIMPLES|OTROS PREP....,895.32
620,VITAMINAS A+D ASOCIACIONES SIMPLES|PRODUCTOS T...,87.88
621,VITAMINAS A+D ASOCIACIONES SIMPLES|REGULADORES...,2324.86
622,VITAMINAS A+D ASOCIACIONES SIMPLES|SUPLEMENTOS...,460.95


In [27]:
pair_analysis

,COD_BODEGA,CLUSTER_bdg,CAT 4_A_bdg,CAT 4_B_bdg,NRO_FACTURAS_CAT4_A_bdg,NRO_FACTURAS_CAT4_B_bdg,NRO_FACTURAS_PAIR_bdg,NRO_FACTURAS_TOTAL_BDG,support_cat4_A_bdg,support_cat4_B_bdg,...,support_AB_clst,confidence_AB_A_clst,lift_ABA_B_clst,new_support_B,support_gap_B,new_tickets,tkt_avg_B,opportunity_B,validation,pairs
0,B101,TRADICIONAL-B,ANTIRREUMATICOS NO ESTEROIDEOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,3435,2908,373,23689,0.145004,0.122757,...,0.016066,0.104015,0.919713,0.118067,-0.004690,-16.110630,16.274082,-1048.742852,0,ANTIRREUMATICOS NO ESTEROIDEOS|ANALGESICOS NO ...
1,B101,TRADICIONAL-B,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIGRIPALES EXC.ANTIINFARINGITIS,3435,2620,361,23689,0.145004,0.110600,...,0.018486,0.119678,0.950268,0.110595,-0.000005,-0.017504,19.746538,-1.382595,0,ANTIRREUMATICOS NO ESTEROIDEOS|ANTIGRIPALES EX...
2,B101,TRADICIONAL-B,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIBACTERIANOS PENICILINAS AMPLIO ESPECTRO,3435,593,197,23689,0.145004,0.025033,...,0.007803,0.050518,2.002788,0.028636,0.003603,12.375577,37.023288,1832.738163,1,ANTIRREUMATICOS NO ESTEROIDEOS|ANTIBACTERIANOS...
3,B101,TRADICIONAL-B,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIHISTAMINICOS-ANTIALERGICOS,3435,655,86,23689,0.145004,0.027650,...,0.002641,0.017097,0.717800,0.034879,0.007229,24.833016,23.471145,2331.437251,0,ANTIRREUMATICOS NO ESTEROIDEOS|ANTIHISTAMINICO...
4,B101,TRADICIONAL-B,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIACIDOS SOLOS,3435,1052,83,23689,0.145004,0.044409,...,0.002564,0.016601,0.485839,0.049735,0.005326,18.294459,10.649401,779.300122,0,ANTIRREUMATICOS NO ESTEROIDEOS|ANTIACIDOS SOLOS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183572,B901,MEDIANA-B,OTRAS VITAMINAS SOLAS Y COMBINADAS,ANALGESICOS NARCOTICOS,159,281,1,23621,0.006731,0.011896,...,0.000105,0.008318,0.554427,0.011345,-0.000551,-0.087627,42.676192,-14.958394,0,OTRAS VITAMINAS SOLAS Y COMBINADAS|ANALGESICOS...
183573,B901,MEDIANA-B,OTRAS VITAMINAS SOLAS Y COMBINADAS,ANGIOTENSINA-II ANTAGONISTAS SOLO,159,275,1,23621,0.006731,0.011642,...,0.000165,0.013094,0.653064,0.009632,-0.002011,-0.319689,47.038000,-60.150182,0,OTRAS VITAMINAS SOLAS Y COMBINADAS|ANGIOTENSIN...
183574,B901,MEDIANA-B,OTRAS VITAMINAS SOLAS Y COMBINADAS,ANTIBACTERIANOS PENICILINAS MEDIO Y REDUCIDO E...,159,194,1,23621,0.006731,0.008213,...,0.000026,0.002059,0.345747,0.018193,0.009980,1.586743,50.395619,319.859646,0,OTRAS VITAMINAS SOLAS Y COMBINADAS|ANTIBACTERI...
183575,B901,MEDIANA-B,OTRAS VITAMINAS SOLAS Y COMBINADAS,ANTIESPASMODICOS,159,199,1,23621,0.006731,0.008425,...,0.000116,0.009224,1.152450,0.005458,-0.002967,-0.471713,25.786784,-48.655876,0,OTRAS VITAMINAS SOLAS Y COMBINADAS|ANTIESPASMO...


In [29]:
pair_analysis.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\opportunity_analysis.xlsx", index=False)

In [48]:
best_opportunities.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\best_opportunities.xlsx", index=False)